In [39]:
import tensorflow as tf
import cv2
import numpy as np
import os
import random
from collections import defaultdict
from tqdm import tqdm
import sys

# Add parent directory to path for utility imports
parent_dir = os.path.abspath('..')
sys.path.insert(0, parent_dir)

from eval_NN import get_seq_length, crop_to_bbox

# Output root directory for Grad-CAM results
OUTPUT_ROOT = (
    "/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/"
    "polypclassificationmi/results/gradcam"
)

class HeatmapCreator(object):
    def __init__(self, model, sequences_fn, output_root=OUTPUT_ROOT, classes=[0,1]):
        # Unpack and prepare the model for Grad-CAM
        model = self._unpack_model(model)
        conv_layer = self._find_target_layer(model)
        # Build a new model that outputs conv feature maps and final predictions
        self.model = tf.keras.Model(
            inputs=model.input,
            outputs=[model.get_layer(conv_layer).output, model.output]
        )
        # Store input size (height, width)
        self.input_size = tuple(model.input.shape.as_list()[1:3])

        self.sequences_fn = sequences_fn
        self.output_folder = output_root
        self.classes = classes
        os.makedirs(self.output_folder, exist_ok=True)

    def _unpack_model(self, model):
        cfg = model.get_config()
        layers = cfg['layers']
        new_layers = []
        weight_locations = {}
        # Flatten any nested Functional blocks so we can rebuild a single flat model
        while layers:
            layer = layers.pop(0)
            if layer['class_name'] == 'Functional':
                inner = layer['config']['layers']
                for l in inner:
                    weight_locations[l['name']] = layer['name']
                # Rewire inputs and outputs
                inner = inner[1:]
                inner[0]['inbound_nodes'] = layer['inbound_nodes']
                outbound = layer['config']['output_layers'][0]
                outbound.append({})
                layers[0]['inbound_nodes'] = [[outbound,],]
                layers = inner + layers
            else:
                new_layers.append(layer)
        cfg['layers'] = new_layers

        # Rebuild model from flattened config
        new_model = tf.keras.Model().from_config(cfg)
        # Transfer weights from original into the new flat model
        for layer in new_model.layers:
            name = layer.name
            if name in weight_locations:
                orig = model.get_layer(weight_locations[name])
                # Drill down through nested names if needed
                sub_names = [name]
                while weight_locations.get(sub_names[0]):
                    sub_names.insert(0, weight_locations[sub_names[0]])
                for sub in sub_names[1:]:
                    orig = orig.get_layer(sub)
                layer.set_weights(orig.get_weights())
            else:
                layer.set_weights(model.get_layer(name).get_weights())
        return new_model

    def _find_target_layer(self, model):
        # Find the last convolutional (4D) layer for Grad-CAM
        for l in reversed(model.layers):
            if len(l.output_shape) == 4:
                return l.name
        raise ValueError("No 4D layer found in model.")

    def _preprocess(self, img_path, ann_path):
        # Load image and mask
        img = cv2.imread(img_path)
        ann = cv2.imread(ann_path, 0)
        # Resize mask to image dimensions
        ann = cv2.resize(ann, (img.shape[1], img.shape[0]))
        # Crop around mask bounding box, then resize to network input
        crop = crop_to_bbox(img, ann, padding=0.2, output_size=self.input_size)
        x = crop.astype(np.float32)
        return tf.expand_dims(x, axis=0)

    def create_frames(self):
        # 1) Read file lines: img_path, ann_path, label, polyp_id
        with open(self.sequences_fn, 'r') as f:
            entries = [line.strip().split() for line in f if line.strip()]

        # 2) Group entries by sequence name (folder name)
        seq_dict = defaultdict(list)
        for img_fn, ann_fn, lbl, pid in entries:
            seq_name = os.path.basename(os.path.dirname(img_fn))
            seq_dict[seq_name].append((img_fn, ann_fn, int(lbl)))

        # 3) Randomly pick 5 sequences (or fewer if not enough)
        all_seqs = list(seq_dict.keys())
        chosen_seqs = random.sample(all_seqs, min(5, len(all_seqs)))

        # 4) For each chosen sequence, pick up to 10 random frames
        for seq_name in chosen_seqs:
            frames = seq_dict[seq_name]
            chosen_frames = random.sample(frames, min(10, len(frames)))

            seq_out = os.path.join(self.output_folder, seq_name)
            os.makedirs(seq_out, exist_ok=True)

            # 5) Generate Grad-CAM for each selected frame
            for idx, (img_fn, ann_fn, label) in enumerate(chosen_frames):
                if label not in self.classes:
                    continue

                # Preprocess and compute Grad-CAM
                img_tensor = self._preprocess(img_fn, ann_fn)
                with tf.GradientTape() as tape:
                    conv_outs, preds = self.model(img_tensor)
                    cls = tf.argmax(preds[0]).numpy()
                    loss = preds[:, cls]
                grads = tape.gradient(loss, conv_outs)

                # Compute guided gradients
                guided = (
                    tf.cast(conv_outs > 0, tf.float32) *
                    tf.cast(grads > 0, tf.float32) *
                    grads
                )
                weights = tf.reduce_mean(guided, axis=(1,2))
                cam = tf.reduce_sum(tf.multiply(weights, conv_outs[0]), axis=-1).numpy()
                cam = np.maximum(cam, 0)
                cam /= (cam.max() + 1e-8)

                # Create cropped heatmap overlay
                orig_crop = cv2.imread(img_fn)
                mask_crop = cv2.imread(ann_fn, 0)
                mask_crop = cv2.resize(mask_crop, (orig_crop.shape[1], orig_crop.shape[0]))
                contours, _ = cv2.findContours(mask_crop, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                heat_overlay = orig_crop.copy()
                for cnt in contours:
                    x, y, w, h = cv2.boundingRect(cnt)
                    heat = cv2.resize((cam * 255).astype(np.uint8), (w, h))
                    heat = cv2.applyColorMap(heat, cv2.COLORMAP_VIRIDIS)
                    heat_overlay[y:y+h, x:x+w] = cv2.addWeighted(
                        orig_crop[y:y+h, x:x+w], 0.6, heat, 0.4, 0
                    )

                # Create full-size annotated image
                full = cv2.imread(img_fn)
                mask_full = cv2.imread(ann_fn, 0)
                mask_full = cv2.resize(mask_full, (full.shape[1], full.shape[0]))
                red_layer = np.zeros_like(full, dtype=np.uint8)
                red_layer[..., 2] = 255  # red channel
                alpha = (mask_full > 0).astype(np.float32) * 0.4  # 40% opacity
                for c in range(3):
                    full[..., c] = (
                        full[..., c] * (1 - alpha) +
                        red_layer[..., c] * alpha
                    ).astype(np.uint8)

                # Combine side by side (heatmap crop | full annotated)
                h_full, w_full = full.shape[:2]
                new_width = int(h_full * heat_overlay.shape[1] / heat_overlay.shape[0])
                heat_resized = cv2.resize(heat_overlay, (new_width, h_full))
                combined = np.concatenate([heat_resized, full], axis=1)

                out_path = os.path.join(seq_out, f"{seq_name}_{idx:02d}_sidebyside.png")
                cv2.imwrite(out_path, combined)

        print(f"Grad-CAM + full-size annotated frames saved under: {self.output_folder}")

    def create_videos(self):
        # (Optional) Implement video creation if needed
        pass

if __name__ == '__main__':
    # Load the trained model (no compilation)
    model_name = (
        'hypVSadn_HDall2023_efficientnet_0_regularized0.0_256x256_'
        '1in_nf64_bnTrue_fcdo0.0_balancedTrue_loss_fl_gamma1.0_sgd_5fold0_best.h5'
    )
    model_path = os.path.join(
        '/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/'
        'polypclassificationmi/code/data/snapshots/all',
        model_name
    )
    model = tf.keras.models.load_model(model_path, compile=False)
    model.summary()

    # Path to your updated sequences file
    sequences_file = (
        '/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/'
        'polypclassificationmi/data/imagesets_characterisation/'
        'test_update.txt'
    )
    mc = HeatmapCreator(model, sequences_file, output_root=OUTPUT_ROOT, classes=[0, 1])
    mc.create_frames()


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 256, 256, 3)]     0         
                                                                 
 efficientnetb0 (Functional  (None, 1280)              4049571   
 )                                                               
                                                                 
 flatten (Flatten)           (None, 1280)              0         
                                                                 
 dense (Dense)               (None, 2)                 2562      
                                                                 
Total params: 4052133 (15.46 MB)
Trainable params: 2562 (10.01 KB)
Non-trainable params: 4049571 (15.45 MB)
_________________________________________________________________
Grad-CAM + full-size annotated frames saved under: /DATASERVER/MIC/GENERAL/STUDENTS/a